<a href="https://colab.research.google.com/github/antonbeski0/lstm_market_dynamics/blob/main/trading_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# PRODUCTION TRADING SYSTEM: COALINDIA.NS | VERSION 3.0 (PROD)
# ============================================================
!pip install yfinance scikit-learn torch -q

import warnings, random, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.calibration import calibration_curve
import yfinance as yf

warnings.filterwarnings("ignore")

def set_seeds(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seeds(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = dict(
    TICKER            = "COALINDIA.NS",
    PERIOD            = "5y",
    WINDOW            = 30,
    EPOCHS            = 60,
    BATCH_SIZE        = 32,
    LR                = 1e-3,
    HIDDEN_SIZE       = 64,
    SIGNAL_THRESHOLD  = 0.65,
    KELLY_SAFETY      = 0.25,
    N_FOLDS           = 5
)

def build_features(df):
    c, v = df['Close'], df['Volume']
    out = pd.DataFrame(index=df.index)
    out['Ret1'] = c.pct_change()
    out['RSI'] = pd.Series(100 - (100 / (1 + (c.diff().clip(lower=0).rolling(14).mean() / (-c.diff().clip(upper=0).rolling(14).mean() + 1e-9)))), index=df.index) / 100.0
    out['Vol_Rel'] = v / (v.rolling(20).mean() + 1e-9)
    out['Mom'] = c.pct_change(10)
    out['BB_Width'] = (c.rolling(20).std() * 4) / (c.rolling(20).mean() + 1e-9)
    out['ATR_Norm'] = ((df['High']-df['Low']).rolling(14).mean()) / (c + 1e-9)
    out['MA_Dist'] = (c / (c.rolling(50).mean() + 1e-9)) - 1
    fwd_ret = out['Ret1'].shift(-1)
    out['Target'] = 0
    out.loc[fwd_ret > 0.003, 'Target'] = 1
    out.loc[fwd_ret < -0.003, 'Target'] = 2
    return out.dropna()

df_raw = yf.download(CFG['TICKER'], period=CFG['PERIOD'], auto_adjust=True, progress=False)
if isinstance(df_raw.columns, pd.MultiIndex): df_raw.columns = df_raw.columns.get_level_values(0)
df_feat = build_features(df_raw)
FEATURES = [c for c in df_feat.columns if c != 'Target']

class TradingGRU(nn.Module):
    def __init__(self, in_d, hidden):
        super().__init__()
        self.gru = nn.GRU(in_d, hidden, 1, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x):
        _, h = self.gru(x)
        return self.head(h[-1])

def train_and_predict(X_train, y_train, X_test):
    scaler = RobustScaler()
    X_tr_scaled = scaler.fit_transform(X_train)
    X_te_scaled = scaler.transform(X_test)
    def seq(X, y):
        return [(torch.FloatTensor(X[i:i+CFG['WINDOW']]), torch.LongTensor([y[i+CFG['WINDOW']-1]])[0]) for i in range(len(X)-CFG['WINDOW'])]
    model = TradingGRU(len(FEATURES), CFG['HIDDEN_SIZE']).to(device)
    opt = optim.Adam(model.parameters(), lr=CFG['LR'])
    loader = DataLoader(seq(X_tr_scaled, y_train), batch_size=CFG['BATCH_SIZE'], shuffle=True)
    for _ in range(CFG['EPOCHS']):
        model.train()
        for xb, yb in loader: opt.zero_grad(); nn.CrossEntropyLoss()(model(xb.to(device)), yb.to(device)).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        X_te_input = np.array([X_te_scaled[i:i+CFG['WINDOW']] for i in range(len(X_te_scaled)-CFG['WINDOW'])])
        logits = model(torch.FloatTensor(X_te_input).to(device))
        return torch.softmax(logits, dim=1).cpu().numpy()

print(f"SYSTEM: {CFG['TICKER']} | GRU Classifier | Walk-Forward")
fold_size = len(df_feat) // (CFG['N_FOLDS'] + 1)
all_probs, all_actuals, fold_wins = [], [], 0

for fold in range(CFG['N_FOLDS']):
    tr_end = fold_size * (fold + 1)
    te_end = tr_end + fold_size
    train_data = df_feat.iloc[:tr_end]
    test_data = df_feat.iloc[tr_end:te_end]
    if len(test_data) <= CFG['WINDOW']:
        print(f"Fold {fold+1}: Skipped (too small)"); continue

    probs = train_and_predict(train_data[FEATURES].values, train_data['Target'].values, test_data[FEATURES].values)
    actuals = test_data['Target'].values[CFG['WINDOW']:]
    preds = np.argmax(probs, axis=1)
    fold_acc = accuracy_score(actuals, preds)
    up_mask = preds == 1
    up_prec = (actuals[up_mask] == 1).mean() if up_mask.sum() > 0 else 0
    is_win = up_prec >= 0.50
    if is_win: fold_wins += 1
    print(f"Fold {fold+1}: Acc={fold_acc:.1%} | UP Prec={up_prec:.2f} | {'✅ WIN' if is_win else '❌ MISS'}")
    all_probs.append(probs); all_actuals.append(actuals)

last_fold_len = len(all_actuals[-1])
flat_probs = np.concatenate(all_probs)
flat_actuals = np.concatenate(all_actuals)

print("\nOUT-OF-SAMPLE CLASSIFICATION REPORT")
print(classification_report(flat_actuals, np.argmax(flat_probs, axis=1), target_names=['Flat','Up','Down']))

calib_tr_p = flat_probs[:-last_fold_len]
calib_tr_y = (flat_actuals[:-last_fold_len] == 1).astype(int)
calib_val_p = flat_probs[-last_fold_len:]
calib_val_y = (flat_actuals[-last_fold_len:] == 1).astype(int)

calibrator = LogisticRegression().fit(calib_tr_p[:, 1].reshape(-1,1), calib_tr_y)
prob_true, prob_pred = calibration_curve(calib_val_y, calib_val_p[:, 1], n_bins=5)

print("\nCalibration Check (Fold 5 OOS):")
for pt, pp in zip(prob_pred, prob_true):
    gap = abs(pt - pp)
    print(f"  Predicted {pp:.0%} -> Actual {pt:.0%} {'✅' if gap < 0.10 else '⚠️'}")

X_all, y_all = df_feat[FEATURES].values, df_feat['Target'].values
final_probs = train_and_predict(X_all, y_all, X_all[-(CFG['WINDOW']+2):])
raw_up_prob = final_probs[-1, 1]
calibrated_up_prob = calibrator.predict_proba([[raw_up_prob]])[0, 1]

up_prec_overall = (flat_actuals[np.argmax(flat_probs, axis=1) == 1] == 1).mean()
quality = ((min(calibrated_up_prob, 0.8)/0.8)*4 + (min(up_prec_overall, 0.6)/0.6)*3 + (fold_wins/CFG['N_FOLDS'])*3)

print(f"\n{'='*40}\n TOMORROW'S PREDICTION\n{'='*40}")
print(f" Raw Softmax (UP)   : {raw_up_prob:.1%}")
print(f" Calibrated P(UP)   : {calibrated_up_prob:.1%}")
print(f" Signal Quality     : {quality:.1f} / 10")
if calibrated_up_prob > CFG['SIGNAL_THRESHOLD']:
    kelly = max(0, calibrated_up_prob*2 - 1) * CFG['KELLY_SAFETY']
    print(f" SIGNAL: BUY | Size: {kelly:.1%} of Capital")
else:
    print(" SIGNAL: HOLD CASH (Confidence below threshold)")
print(f"{'='*40}")

SYSTEM: COALINDIA.NS | GRU Classifier | Walk-Forward
Fold 1: Acc=44.6% | UP Prec=0.49 | ❌ MISS
Fold 2: Acc=36.3% | UP Prec=0.45 | ❌ MISS
Fold 3: Acc=36.3% | UP Prec=0.43 | ❌ MISS
Fold 4: Acc=43.5% | UP Prec=0.44 | ❌ MISS
Fold 5: Acc=42.9% | UP Prec=0.51 | ✅ WIN

OUT-OF-SAMPLE CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Flat       0.34      0.11      0.17       174
          Up       0.46      0.46      0.46       363
        Down       0.37      0.51      0.43       303

    accuracy                           0.41       840
   macro avg       0.39      0.36      0.35       840
weighted avg       0.40      0.41      0.39       840


Calibration Check (Fold 5 OOS):
  Predicted 35% -> Actual 9% ⚠️
  Predicted 54% -> Actual 29% ⚠️
  Predicted 50% -> Actual 53% ✅
  Predicted 45% -> Actual 70% ⚠️
  Predicted 44% -> Actual 94% ⚠️

 TOMORROW'S PREDICTION
 Raw Softmax (UP)   : 45.1%
 Calibrated P(UP)   : 43.5%
 Signal Quality     : 5.1 / 10
 SIGNAL: HOLD